# Relevance Scores from the Item 1 Sentences

| Item | Description |
|---|---|
| Goal | Two relevance scores per firm built from its raw Item 1 sentences and the sentences' cosine similarity to the topic vector c (T3.6.5), using the thresholds fixed in T3.6.5: (1) binary, whether any sentence is at or above the threshold; (2) share, the sentences at or above the threshold over all Item 1 sentences. A third score, the mean cosine of the sentences at or above the threshold, was computed in an earlier version and dropped on 2026-10-25 because it varies only within the narrow band above the threshold and tracks nothing. |
| Input | `<scratch>/T3.6 - Firm Sentences/Processed/sentence_similarities.parquet` and `sentences.parquet` (T3.6.1, T3.6.5); `output/T3.6.5/similarity_threshold.csv`; `output/T3.5.1/topic_definition.csv`; `output/T3.5.3/nearest_statements.csv`; `output/T3.4.1/filings.csv`; `output/T3.6.6/` and `output/T3.6.7/` (the share must reproduce them). |
| Output | `output/T3.8.1/score_sentences.csv`: one row per firm, sample, topic vector, and measure. |
| Unit | A firm (gvkey) within a sample; the sentence is the unit the scores are counted over. |
| Sample | All years (18,779 filings, 1,058 firms) and fiscal years 2010, 2015, 2020 (2,072 filings, 841 firms), as flagged in T3.4.1. |
| Topic side | c = the embedded topic definition (T3.5.1) or the centroid of the 374 Topic 68 hearing statements (T3.5.3). The policy summary (T3.5.7) has no sentence similarities and is used only in T3.8.2. |
| Example | Meta Platforms, worked through step by step. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for the surrounding steps.


## Step 1. Import packages


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


## Step 2. Define file locations and settings


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.8.1"  # the identifier names this notebook's output folder
TEMP_OUTPUT = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output"
OUTPUT_FOLDER = TEMP_OUTPUT / NOTEBOOK_NAME
SENTENCES_FOLDER = SCRATCH_FOLDER / "T3.6 - Firm Sentences" / "Processed"
SIMILARITIES_FILE = SENTENCES_FOLDER / "sentence_similarities.parquet"
SENTENCES_FILE = SENTENCES_FOLDER / "sentences.parquet"
THRESHOLD_FILE = TEMP_OUTPUT / "T3.6.5/similarity_threshold.csv"
DEFINITION_FILE = TEMP_OUTPUT / "T3.5.1/topic_definition.csv"
NEAREST_STATEMENTS_FILE = TEMP_OUTPUT / "T3.5.3/nearest_statements.csv"
FILINGS_FILE = TEMP_OUTPUT / "T3.4.1/filings.csv"
SHARE_FILES = {"definition": TEMP_OUTPUT / "T3.6.6/score_sentence_share_definition.csv", "centroid": TEMP_OUTPUT / "T3.6.7/score_sentence_share_centroid.csv"}

TOPIC_VECTORS = ["definition", "centroid"]  # the two vectors with sentence similarities
SAMPLES = {"all_years": "in_all_years", "three_years": "in_three_years"}  # sample name to its flag in filings.csv
MEASURES = ["binary", "share"]
EXAMPLE_FIRM = "META PLATFORMS INC"
N_EXAMPLE_SENTENCES = 8  # of the example firm's sentences above the threshold shown
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Outputs: {OUTPUT_FOLDER}")


Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.8.1


## Step 3. The topic side: what the sentences are compared with

Two vectors c, both from T3.5. The **definition vector** is the embedding of the codebook label and definition of Topic 68. The **centroid vector** is the mean of the embeddings of the 374 hearing statements coded to Topic 68 (support and concern together), scaled to unit length; the statements nearest to it show what it stands for. Each vector has its own threshold, fixed by hand in T3.6.5.


In [3]:
definition = pd.read_csv(DEFINITION_FILE).iloc[0]
print(f"DEFINITION VECTOR = embedding of this text ({definition['n_tokens']} tokens):")
print(f'  "{definition["topic_text"]}"')
nearest = pd.read_csv(NEAREST_STATEMENTS_FILE).head(3)
print("\nCENTROID VECTOR = mean embedding of the 374 Topic 68 statements; the three statements nearest to it:")
for statement in nearest.itertuples(index=False):
    print(f"  [{statement.hearing_year}, {statement.stance}, cosine to centroid {statement.similarity_to_centroid:.3f}] {statement.text[:300]}{'...' if len(statement.text) > 300 else ''}")
thresholds = pd.read_csv(THRESHOLD_FILE).set_index("vector")
THRESHOLD = {vector: float(thresholds.loc[vector, "threshold"]) for vector in TOPIC_VECTORS}
print(f"\nTHRESHOLDS (T3.6.5, {thresholds['decided_by'].iloc[0]}): {THRESHOLD}")


DEFINITION VECTOR = embedding of this text (36 tokens):
  "consumer data privacy legislation: This topic encompasses federal and state laws, proposed bills, and regulatory frameworks specifically addressing the collection, use, and protection of consumer personal information."

CENTROID VECTOR = mean embedding of the 374 Topic 68 statements; the three statements nearest to it:
  [2008, concern, cosine to centroid 0.879] I think that the issue here of privacy, who is collecting what kind of information, how does that information exist in what identifiable form or how might that be used, who is it sold to, does the consumer know, all of those issues I think are very, very important. And the issues that surround them ...
  [2019, support, cosine to centroid 0.876] Well, I happen to believe that this is one of the areas, maybe not many areas in this next couple of years, that we ought to be able to come together around in a bipartisan way and come up with a national data privacy law that c

## Step 4. The firm side: the Item 1 sentences and their similarities

Every sentence of every filing's Item 1 (T3.6.1) with its cosine similarity to each vector (T3.6.5), joined to the sample flags.


In [4]:
similarities = pd.read_parquet(SIMILARITIES_FILE)
sentences = pd.read_parquet(SENTENCES_FILE)
assert len(similarities) == len(sentences) and (similarities["row"].to_numpy() == np.arange(len(sentences))).all()
sentences["cos_definition"] = similarities["sim_definition"].to_numpy()
sentences["cos_centroid"] = similarities["sim_centroid"].to_numpy()
del similarities
filings = pd.read_csv(FILINGS_FILE, dtype={"gvkey": str, "cik": str})
sentences = sentences.merge(filings[["accessionNo", "conm", "in_all_years", "in_three_years"]], on="accessionNo", how="left", validate="many_to_one")
assert sentences["in_all_years"].notna().all()
for vector in TOPIC_VECTORS:
    sentences[f"above_{vector}"] = sentences[f"cos_{vector}"].ge(THRESHOLD[vector])
    assert int(sentences[f"above_{vector}"].sum()) == int(thresholds.loc[vector, "n_sentences_at_or_above"])
print(f"Sentences: {len(sentences):,} from {sentences['accessionNo'].nunique():,} filings and {sentences['gvkey'].nunique():,} firms")
print({vector: f"{int(sentences[f'above_{vector}'].sum()):,} sentences at or above {THRESHOLD[vector]}" for vector in TOPIC_VECTORS})


Sentences: 5,647,505 from 18,779 filings and 1,058 firms
{'definition': '6,183 sentences at or above 0.67', 'centroid': '3,956 sentences at or above 0.75'}


## Step 5. How the two scores are computed

For firm $i$ in a sample, let $S_i$ be all Item 1 sentences of the firm's filings in that sample, $e_k$ the embedding of sentence $k$, $c$ the topic vector, and $\tau_c$ its threshold. The sentences that count as about the topic are

$$A_i = \{\, k \in S_i : \cos(e_k, c) \ge \tau_c \,\}.$$

1. **Binary**: 1 if the firm has at least one such sentence, else 0.
$$\text{binary}_i = \mathbb{1}[\,|A_i| > 0\,]$$

2. **Share**: the sentences about the topic over all the firm's Item 1 sentences (the same number as T3.6.6 and T3.6.7).
$$\text{share}_i = \frac{|A_i|}{|S_i|}$$

In words: pool the firm's sentences over its filings in the sample, mark the ones at or above the threshold, then take (1) whether there is any and (2) their fraction.


In [5]:
def firm_scores(frame, vector):
    grouped = frame.groupby("gvkey")
    out = grouped.agg(conm=("conm", "first"), n_filings=("accessionNo", "nunique"), n_sentences=("sentence", "size"),
                      n_above_threshold=(f"above_{vector}", "sum"))
    out["binary"] = out["n_above_threshold"].gt(0).astype(int)
    out["share"] = out["n_above_threshold"] / out["n_sentences"]
    return out.reset_index()


wide_rows = []
for sample_name, flag in SAMPLES.items():
    in_sample = sentences.loc[sentences[flag].eq(1)]
    for vector in TOPIC_VECTORS:
        wide_rows.append(firm_scores(in_sample, vector).assign(sample=sample_name, topic_vector=vector, threshold=THRESHOLD[vector]))
wide = pd.concat(wide_rows, ignore_index=True)
assert wide["share"].between(0, 1).all()
summary = wide.groupby(["sample", "topic_vector"]).agg(firms=("gvkey", "size"), firms_with_any=("binary", "sum"),
                                                      share_median_if_any=("share", lambda s: s[s > 0].median()))
display(summary.round(4))


firms  firms_with_any  share_median_if_any
sample      topic_vector                                            
all_years   centroid       1058             265               0.0016
            definition     1058             306               0.0021
three_years centroid        841             162               0.0033
            definition      841             192               0.0033

## Step 6. Check the share against T3.6.6 and T3.6.7

The share here must equal the 1A score those notebooks saved, firm by firm.


In [6]:
for vector, path in SHARE_FILES.items():
    earlier = pd.read_csv(path, dtype={"gvkey": str})
    merged = wide.loc[wide["topic_vector"].eq(vector)].merge(earlier, on=["gvkey", "sample"], suffixes=("", "_earlier"), validate="one_to_one")
    assert len(merged) == len(earlier), vector
    assert np.allclose(merged["share"], merged["score"]) and merged["n_sentences"].eq(merged["n_sentences_earlier"]).all(), vector
    print(f"{vector}: share equals the T3.6.{6 if vector == 'definition' else 7} score for all {len(merged):,} firm-sample rows")


definition: share equals the T3.6.6 score for all 1,899 firm-sample rows
centroid: share equals the T3.6.7 score for all 1,899 firm-sample rows


## Step 7. Worked example: Meta Platforms

The firm's sentences in each sample, the ones above each threshold, and the two scores computed from them. Then its sentences above the definition threshold in the three-year sample, with their cosines.


In [7]:
example = wide.loc[wide["conm"].eq(EXAMPLE_FIRM)]
assert example["gvkey"].nunique() == 1, "the example firm name must match one gvkey"
example_gvkey = example["gvkey"].iloc[0]
print(f"{EXAMPLE_FIRM} (gvkey {example_gvkey})")
for row in example.sort_values(["sample", "topic_vector"]).itertuples(index=False):
    print(f"  {row.sample:12s} {row.topic_vector:10s} filings {row.n_filings:2d}, sentences {row.n_sentences:5d}, above {row.threshold}: {row.n_above_threshold:3d}"
          f"  ->  binary {row.binary}, share {row.n_above_threshold}/{row.n_sentences} = {row.share:.4f}")
shown = sentences.loc[sentences["gvkey"].eq(example_gvkey) & sentences["in_three_years"].eq(1) & sentences["above_definition"]]
shown = shown.sort_values("cos_definition", ascending=False)
print(f"\n{EXAMPLE_FIRM}, three-year sample: {len(shown)} sentences at or above the definition threshold; the top {min(N_EXAMPLE_SENTENCES, len(shown))}:")
for row in shown.head(N_EXAMPLE_SENTENCES).itertuples(index=False):
    print(f"  [fiscal {row.fyear}, cosine {row.cos_definition:.3f}] {row.sentence[:260]}{'...' if len(row.sentence) > 260 else ''}")


META PLATFORMS INC (gvkey 170617)
  all_years    centroid   filings 14, sentences  1870, above 0.75:  35  ->  binary 1, share 35/1870 = 0.0187
  all_years    definition filings 14, sentences  1870, above 0.67:  37  ->  binary 1, share 37/1870 = 0.0198
  three_years  centroid   filings  2, sentences   183, above 0.75:   5  ->  binary 1, share 5/183 = 0.0273
  three_years  definition filings  2, sentences   183, above 0.67:   5  ->  binary 1, share 5/183 = 0.0273



META PLATFORMS INC, three-year sample: 5 sentences at or above the definition threshold; the top 5:
  [fiscal 2015, cosine 0.742] In particular, we are subject to federal, state, and foreign laws regarding privacy and protection of people's data.
  [fiscal 2020, cosine 0.742] In particular, we are subject to federal, state, and foreign laws regarding privacy and protection of people's data.
  [fiscal 2020, cosine 0.730] These laws and regulations involve matters including privacy, data use, data protection and personal information, rights of publicity, content, intellectual property, advertising, marketing, distribution, data security, data retention and deletion, data locali...
  [fiscal 2015, cosine 0.706] These may involve user privacy and data protection, rights of publicity, content, intellectual property, advertising, marketing, distribution, data security, data retention and deletion, personal information, electronic contracts and other communications, comp...
  [fiscal 2015, c

## Step 8. Save the scores

One long table: one row per firm, sample, topic vector, and measure, with the counts the measure is built from repeated on every row.


In [8]:
long = wide.melt(id_vars=["gvkey", "conm", "sample", "topic_vector", "threshold", "n_filings", "n_sentences", "n_above_threshold"],
                 value_vars=MEASURES, var_name="measure", value_name="score")
long["method"] = "sentences"
long = long[["gvkey", "conm", "method", "sample", "topic_vector", "threshold", "measure", "n_filings", "n_sentences", "n_above_threshold", "score"]]
scores_file = OUTPUT_FOLDER / "score_sentences.csv"
long.sort_values(["sample", "topic_vector", "measure", "gvkey"]).to_csv(scores_file, index=False, float_format="%.6f")
print(f"Saved: {scores_file} ({len(long):,} rows = {wide['gvkey'].nunique():,} firms x samples x {len(TOPIC_VECTORS)} vectors x {len(MEASURES)} measures)")
print(f"Columns: {list(long.columns)}")
assert long["score"].notna().all()


Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.8.1/score_sentences.csv (7,596 rows = 1,058 firms x samples x 2 vectors x 2 measures)
Columns: ['gvkey', 'conm', 'method', 'sample', 'topic_vector', 'threshold', 'measure', 'n_filings', 'n_sentences', 'n_above_threshold', 'score']
